# CV Skill Extraction using Machine Learning

**Course:** Machine Learning

This notebook demonstrates how to extract technical skills from a CV (PDF) using:
- Text extraction from PDF
- Text preprocessing
- TF-IDF feature extraction
- Keyword matching with scoring

In [ ]:
# Install required libraries
# !pip install scikit-learn PyPDF2

In [ ]:
import re
import json
from collections import Counter

# Define known skills (same as the frontend SKILLS list)
KNOWN_SKILLS = [
    'Python', 'Java', 'C++', 'C#', 'JavaScript', 'TypeScript',
    'Go', 'Rust', 'PHP', 'Ruby', 'Swift', 'Kotlin', 'Dart', 'R', 'MATLAB',
    'React', 'Angular', 'Vue.js', 'Next.js', 'Node.js', 'Express.js',
    'Django', 'Flask', 'FastAPI', 'Spring Boot', 'Laravel',
    'TensorFlow', 'PyTorch', 'Keras', 'Scikit-Learn', 'Pandas',
    'NumPy', 'OpenCV', 'Hugging Face', 'NLTK', 'spaCy',
    'SQL', 'MySQL', 'PostgreSQL', 'MongoDB', 'Redis',
    'Firebase', 'Supabase', 'Elasticsearch',
    'AWS', 'Google Cloud', 'Microsoft Azure', 'Docker', 'Kubernetes',
    'Linux', 'Git', 'GitHub Actions',
    'Flutter', 'React Native',
]

print(f'Total known skills: {len(KNOWN_SKILLS)}')

In [ ]:
def extract_text_from_pdf(file_path):
    """Extract text from a PDF file using PyPDF2."""
    try:
        from PyPDF2 import PdfReader
        reader = PdfReader(file_path)
        text = ''
        for page in reader.pages:
            text += page.extract_text() + '\n'
        return text
    except ImportError:
        print('PyPDF2 not installed. Using sample text instead.')
        return None


def preprocess_text(text):
    """Clean and normalize text for skill matching."""
    # Lowercase
    text = text.lower()
    # Remove special characters except common tech symbols
    text = re.sub(r'[^\w\s.+#\/\-]', ' ', text)
    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def extract_skills(text, known_skills):
    """Match known skills in the text with frequency-based confidence."""
    processed = preprocess_text(text)
    results = {}
    
    for skill in known_skills:
        # Escape special regex characters in skill name
        pattern = re.escape(skill.lower())
        matches = re.findall(r'\b' + pattern + r'\b', processed)
        
        if matches:
            count = len(matches)
            # Confidence: scales from 0.5 (1 mention) to 1.0 (5+ mentions)
            confidence = min(0.5 + (count * 0.1), 1.0)
            results[skill] = {
                'count': count,
                'confidence': round(confidence, 2)
            }
    
    return results

In [ ]:
# Demo with sample CV text
sample_cv = """
Ahmed Mohamed - Software Engineer

Technical Skills:
- Programming: Python, JavaScript, TypeScript, C++
- Frameworks: React, Node.js, Django, Flask
- Machine Learning: TensorFlow, Scikit-Learn, Pandas, NumPy
- Databases: PostgreSQL, MongoDB, Redis
- DevOps: Docker, Git, Linux, AWS

Experience:
- Built a machine learning pipeline using Python and TensorFlow
- Developed a full-stack web application with React and Node.js
- Deployed microservices using Docker and AWS
- Worked with PostgreSQL and MongoDB databases
"""

# Extract skills
detected = extract_skills(sample_cv, KNOWN_SKILLS)

# Sort by confidence
sorted_skills = sorted(detected.items(), key=lambda x: x[1]['confidence'], reverse=True)

print('Detected Skills:')
print('-' * 50)
for skill, info in sorted_skills:
    print(f'  {skill:20s} | Count: {info["count"]} | Confidence: {info["confidence"]}')

print(f'\nTotal skills detected: {len(detected)}')

In [ ]:
# TF-IDF Analysis (bonus: demonstrates ML concept)
try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    
    # Treat each skill as a "document" and the CV as a query
    documents = [skill.lower() for skill in KNOWN_SKILLS]
    documents.append(preprocess_text(sample_cv))
    
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(documents)
    
    # Get feature names
    features = vectorizer.get_feature_names_out()
    
    # Show top TF-IDF terms from the CV
    cv_vector = tfidf_matrix[-1].toarray()[0]
    top_indices = cv_vector.argsort()[-15:][::-1]
    
    print('Top TF-IDF terms from CV:')
    for idx in top_indices:
        if cv_vector[idx] > 0:
            print(f'  {features[idx]:20s} | TF-IDF: {cv_vector[idx]:.4f}')
except ImportError:
    print('scikit-learn not installed. Skipping TF-IDF analysis.')

In [ ]:
# Output in the format expected by the frontend
output = {
    'skills_detected': [skill for skill, _ in sorted_skills]
}

print(json.dumps(output, indent=2))